In [29]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

In [30]:
import os
print(os.getcwd())

c:\Users\annus\OneDrive\Desktop\Customer-360-Analytics\Python


In [31]:
import pandas as pd

df = pd.read_csv("../Dataset/customer_data.csv")

df.head()

,Sum of Monetary,city,Sum of age,gender,Sum of RFM_Score,Segment,customer_segment,Sum of Frequency,Sum of Recency,customer_id
0,557684,Delhi,58,Male,3,At Risk,Budget,6,123,C0001
1,1455050,Mumbai,33,Male,11,Loyal Customers,Budget,16,61,C0002
2,856505,Hyderabad,61,Male,8,Potential Loyal,Regular,12,61,C0003
3,737491,Mumbai,19,Male,10,Loyal Customers,Premium,11,4,C0004
4,724011,Hyderabad,56,Male,4,At Risk,Premium,7,108,C0005


In [32]:
df.columns

Index(['Sum of Monetary', 'city', 'Sum of age', 'gender', 'Sum of RFM_Score',
       'Segment', 'customer_segment', 'Sum of Frequency', 'Sum of Recency',
       'customer_id'],
      dtype='object')

In [33]:
df = df.rename(columns={
    "Sum of Monetary": "Monetary",
    "Sum of age": "age",
    "Sum of RFM_Score": "RFM_Score",
    "Sum of Frequency": "Frequency",
    "Sum of Recency": "Recency"
})

In [34]:
df.columns

Index(['Monetary', 'city', 'age', 'gender', 'RFM_Score', 'Segment',
       'customer_segment', 'Frequency', 'Recency', 'customer_id'],
      dtype='object')

In [54]:
import numpy as np

threshold = df["Recency"].quantile(0.75)

df["Churn"] = np.where(
    df["Recency"] >= threshold,
    1,
    0
)

In [55]:
df["Churn"].value_counts()

Churn
0    373
1    127
Name: count, dtype: int64

In [56]:
X = df[
[
    "age",
    "Recency",
    "Frequency",
    "Monetary",
    "RFM_Score"
]
]

y = df["Churn"]

In [57]:
X = df[
[
    "age",
    "Recency",
    "Frequency",
    "Monetary",
    "RFM_Score"
]
]

y = df["Churn"]

In [58]:
X.head()

,age,Recency,Frequency,Monetary,RFM_Score
0,58,123,6,557684,3
1,33,61,16,1455050,11
2,61,61,12,856505,8
3,19,4,11,737491,10
4,56,108,7,724011,4


In [59]:
y.value_counts()

Churn
0    373
1    127
Name: count, dtype: int64

Split Training and Testing Data

In [60]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [61]:
print(X_train.shape)
print(X_test.shape)

(400, 5)
(100, 5)


In [62]:
from sklearn.ensemble import RandomForestClassifier

In [63]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [64]:
model.fit(
    X_train,
    y_train
)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [65]:
y_pred = model.predict(X_test)

In [66]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [67]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        75
           1       1.00      1.00      1.00        25

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [68]:
df["Churn_Prediction"] = model.predict(X)

In [69]:
df["Churn_Probability"] = (
    model.predict_proba(X)[:,1] * 100
)

In [70]:
df[
[
"customer_id",
"Churn_Prediction",
"Churn_Probability"
]
].head(10)

,customer_id,Churn_Prediction,Churn_Probability
0,C0001,1,100.0
1,C0002,1,96.0
2,C0003,1,98.0
3,C0004,0,0.0
4,C0005,1,100.0
5,C0006,1,100.0
6,C0007,0,0.0
7,C0008,0,0.0
8,C0009,0,0.0
9,C0010,0,0.0


In [71]:
df["Risk_Level"] = np.where(
    df["Churn_Probability"] >= 70,
    "High Risk",
    np.where(
        df["Churn_Probability"] >= 40,
        "Medium Risk",
        "Low Risk"
    )
)

In [72]:
df["Risk_Level"].value_counts()

Risk_Level
Low Risk     373
High Risk    127
Name: count, dtype: int64

In [73]:
ml_output = df[
[
"customer_id",
"age",
"city",
"customer_segment",
"Recency",
"Frequency",
"Monetary",
"RFM_Score",
"Churn_Prediction",
"Churn_Probability",
"Risk_Level"
]
]

In [74]:
ml_output.head()

,customer_id,age,city,customer_segment,Recency,Frequency,Monetary,RFM_Score,Churn_Prediction,Churn_Probability,Risk_Level
0,C0001,58,Delhi,Budget,123,6,557684,3,1,100.0,High Risk
1,C0002,33,Mumbai,Budget,61,16,1455050,11,1,96.0,High Risk
2,C0003,61,Hyderabad,Regular,61,12,856505,8,1,98.0,High Risk
3,C0004,19,Mumbai,Premium,4,11,737491,10,0,0.0,Low Risk
4,C0005,56,Hyderabad,Premium,108,7,724011,4,1,100.0,High Risk


In [75]:
ml_output.to_csv(
"customer_churn_prediction.csv",
index=False
)

In [76]:
ml_output.to_csv(
"../Dataset/customer_churn_prediction.csv",
index=False
)